### Ce notebook permet de faire une vérification de la qualité des données avant toute consolidation, visualisation ou modélisation.

In [1]:
import pandas as pd
from pathlib import Path

###  Chargement de données

In [2]:
# Chemin d'accès aux données brutes

RAW_DATA_DIR = Path("../..") / "data" / "raw"

# Charger les datasets

files = {
    "idmc": "data_idmc_depuis_2000.csv",
    "solutions": "data_solutions_depuis_2000.csv",
    "decisions": "decisions_asile_depuis_2000.csv",
    "demandes": "demandes_asile_depuis_2000.csv",
    "demographie": "demographie_depuis_2000.csv",
    "pays": "countries.csv"
}


dfs = {
    name: pd.read_csv(RAW_DATA_DIR / filename)
    for name, filename in files.items()
}

for name, df in dfs.items():
    print(f"{name:15} : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

idmc            : 934 lignes × 10 colonnes
solutions       : 21,258 lignes × 13 colonnes
decisions       : 113,929 lignes × 17 colonnes
demandes        : 120,597 lignes × 14 colonnes
demographie     : 116,781 lignes × 24 colonnes
pays            : 232 lignes × 16 colonnes


### Vérifier la qualité de total IDMC

- Cette variable servira à construire la target

In [3]:
idmc = dfs["idmc"].copy()

In [4]:
total = pd.to_numeric(
    idmc["total"],
    errors="coerce"
)

print("Missing :", total.isna().sum())
print("Zéro :", (total == 0).sum())
print("Négatifs :", (total < 0).sum())
print("Minimum :", total.min())
print("Maximum :", total.max())
print("Médiane :", total.median())

Missing : 0
Zéro : 0
Négatifs : 0
Minimum : 2
Maximum : 28000000
Médiane : 204000.0


### Vérification des outliers sans les supprimer

In [5]:
display(
    total.describe(
        percentiles=[
            .01, .05, .25,
            .50, .75, .95, .99
        ]
    )
)

count    9.340000e+02
mean     1.272705e+06
std      3.560060e+06
min      2.000000e+00
1%       1.100000e+02
5%       9.000000e+02
25%      2.500000e+04
50%      2.040000e+05
75%      6.805000e+05
95%      5.614500e+06
99%      2.403700e+07
max      2.800000e+07
Name: total, dtype: float64

In [6]:
idmc_ml = (
    idmc
    .groupby(
        ["year", "coo_id", "coo_iso"],
        as_index=False,
        dropna=False
    )
    .agg(
        idmc_total=("total", "sum")
    )
)

In [7]:
print(
    "Doublons année × pays :",
    idmc_ml.duplicated(
        ["year", "coo_id"]
    ).sum()
)

Doublons année × pays : 0


### Contrôler la future target >15 %

- permet de clacul d'une croissance historique


*Combien d'IDPs ce pays possède-t-il cette année ?* - *« À partir de l'année T, ce pays connaîtra-t-il une hausse supérieure à 15 % à T+1 ? »*

In [35]:
# Trier chronologiquement chaque pays
idmc_ml = (
    idmc_ml
    .sort_values(["coo_id", "year"])
)

# Chercher l'année suivante
idmc_ml["year_t1"] = (
    idmc_ml
    .groupby("coo_id")["year"]
    .shift(-1)
)

# Chercher le stock IDMC de l'observation suivante
idmc_ml["total_t1"] = (
    idmc_ml
    .groupby("coo_id")["idmc_total"]
    .shift(-1)
)


# Vérifier que T+1 est réellement l'année suivante
idmc_ml["valid_t1"] = (
    idmc_ml["year_t1"]
    == idmc_ml["year"] + 1
)

# Définir les lignes sur lesquelles la croissance peut être calculée
valid = (
    idmc_ml["valid_t1"]
    & (idmc_ml["idmc_total"] > 0)
)

# Calculer la croissance entre T et T+1
idmc_ml.loc[valid, "growth_t1"] = (
    (
        idmc_ml.loc[valid, "total_t1"]
        - idmc_ml.loc[valid, "idmc_total"]
    )
    / idmc_ml.loc[valid, "idmc_total"]
)


# Construire la cible ML
idmc_ml.loc[
    valid,
    "hausse_critique_suivante"
] = (
    idmc_ml.loc[valid, "growth_t1"] > 0.15
).astype(int)


# Identifier les stocks IDMC égaux à zéro
zero_base = (
    idmc_ml["idmc_total"] == 0
)

print(
    "Bases IDMC nulles :",
    zero_base.sum()
)


# Permet de connaitre un phénomène de déplacement apparaît là où aucun stock n'était précédemment observé.
idmc_ml["new_emergence"] = (
    (idmc_ml["idmc_total"] == 0)
    & (idmc_ml["total_t1"] > 0)
).astype(int)


# le nombre d'observations de chaque classe
display(
    idmc_ml[
        "hausse_critique_suivante"
    ]
    .value_counts(dropna=False)
)


# Calculer les proportions
display(
    idmc_ml[
        "hausse_critique_suivante"
    ]
    .value_counts(
        normalize=True,
        dropna=False
    ) * 100
)

Bases IDMC nulles : 0


hausse_critique_suivante
0.0    654
1.0    181
NaN     99
Name: count, dtype: int64

hausse_critique_suivante
0.0    70.021413
1.0    19.379015
NaN    10.599572
Name: proportion, dtype: float64

In [34]:
idmc_ml = (
    idmc_ml
    .sort_values(["coo_id", "year"])
)

display(idmc_ml)

,year,coo_id,coo_iso,idmc_total,year_t1,total_t1,valid_t1,growth_t1,hausse_critique_suivante,new_emergence,previous_year,year_gap,transition_annuelle_valide,total_t_1,growth_t,next_year,transition_t1_valide
19,2009,2,AFG,297000,2010.0,352000.0,True,0.185185,1.0,0,NaN,NaN,False,NaN,NaN,2010.0,True
65,2010,2,AFG,352000,2011.0,450000.0,True,0.278409,1.0,0,2009.0,1.0,True,297000.0,0.185185,2011.0,True
112,2011,2,AFG,450000,2012.0,492000.0,True,0.093333,0.0,0,2010.0,1.0,True,352000.0,0.278409,2012.0,True
161,2012,2,AFG,492000,2013.0,631000.0,True,0.282520,1.0,0,2011.0,1.0,True,450000.0,0.093333,2013.0,True
210,2013,2,AFG,631000,2014.0,805000.0,True,0.275753,1.0,0,2012.0,1.0,True,492000.0,0.282520,2014.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
701,2021,263,AB9,15000,2022.0,56000.0,True,2.733333,1.0,0,2020.0,1.0,True,19000.0,-0.210526,2022.0,True
761,2022,263,AB9,56000,2023.0,42000.0,True,-0.250000,0.0,0,2021.0,1.0,True,15000.0,2.733333,2023.0,True
820,2023,263,AB9,42000,2024.0,42000.0,True,0.000000,0.0,0,2022.0,1.0,True,56000.0,-0.250000,2024.0,True
880,2024,263,AB9,42000,2025.0,58000.0,True,0.380952,1.0,0,2023.0,1.0,True,42000.0,0.000000,2025.0,True
